# MUFASA SFT - Gemma 3 1B

Full-parameter SFT on the CPT checkpoint. Unsloth's official Gemma 3 notebook, carried as-is apart from full finetuning, the CPT checkpoint, and the MUFASA dataset.


In [ ]:
# The pinned stack from the CPT notebooks. Deliberately NOT the Unsloth
# template pins (transformers 4.56.2 / trl 0.22.2): the CPT checkpoints were
# produced under transformers 5.5.0, and loading them a major version below
# that risks a silent misload. Everything else below is Unsloth verbatim.
%pip install -qU "unsloth[colab-new,colab-no-deps]==2026.8.19" "unsloth_zoo==2026.8.13" "transformers==5.5.0" "trl==0.24.0" "datasets==4.3.0"


In [ ]:
# ---- paths -------------------------------------------------------------
# The only cell you must edit. Everything below is Unsloth's own code.
import sys
from pathlib import Path

if "google.colab" in sys.modules:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE = Path("/content/drive/MyDrive/mufasa")
else:
    DRIVE = next((p for p in [Path.cwd(), *Path.cwd().parents]
                  if (p / "02-model-engineering").is_dir()), Path.cwd())

# Your CPT output. Full finetuning needs MERGED 16-bit weights, not a LoRA
# adapter folder - see the note under this cell if you only saved the adapter.
CPT_CHECKPOINT = DRIVE / "checkpoints" / "gemma3-1b-cpt-v4" / "merged_16bit"

# The training set from build-training-set.ipynb. LATEST.json names the current
# published run, so this follows a rebuild without editing anything here.
import json

TRAINING_ROOT = DRIVE / "training_set"
_latest = TRAINING_ROOT / "LATEST.json"
if _latest.is_file():
    _run = json.loads(_latest.read_text(encoding="utf-8"))["directory"]
else:
    _run = "runs/0fe2a1ff46c34c5179e0bea7"      # the run published on 2026-08-24
SFT_PARQUET = TRAINING_ROOT / _run / "sft_mixed.parquet"
print("run:", _run)

for path in (CPT_CHECKPOINT, SFT_PARQUET):
    print(("OK     " if path.exists() else "MISSING"), path)

RUN_LABEL = "gemma3-1b-cpt-v4"
MODEL_API = None  # set in the loader cell below


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
run: runs/0fe2a1ff46c34c5179e0bea7
OK      /content/drive/MyDrive/mufasa/checkpoints/gemma3-1b-cpt-v4/merged_16bit
OK      /content/drive/MyDrive/mufasa/training_set/runs/0fe2a1ff46c34c5179e0bea7/sft_mixed.parquet


### If you only saved the LoRA adapter

Full finetuning starts from real weights, so a CPT adapter folder has to be
merged first. Run this once, in a throwaway session, then point
`CPT_CHECKPOINT` at the output:

```python
from unsloth import FastLanguageModel
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "<your CPT lora folder>", max_seq_length = 2048, load_in_4bit = False,
)
model.save_pretrained_merged("<...>/merged_16bit", tokenizer, save_method = "merged_16bit")
```

If your CPT notebook already ran its `SAVE_MERGED` cell, the folder exists and
there is nothing to do.


In [ ]:
from unsloth import FastModel
MODEL_API = FastModel
import torch
max_seq_length = 4096   # p99 of this data is 2,299 tokens; 2048 truncated
                        # 3.3% of examples, and truncation eats the ANSWER

model, tokenizer = FastModel.from_pretrained(
    model_name = str(CPT_CHECKPOINT),
    max_seq_length = max_seq_length, # Choose any for long context!
    load_in_4bit = False,  # 4 bit quantization to reduce memory
    load_in_8bit = False, # [NEW!] A bit more accurate, uses 2x memory
    full_finetuning = True, # SFT, not LoRA - Unsloth's own full-finetuning path
    # Skips the flex_attention path entirely, so the kernel above is
    # never compiled. Try this first; drop the line to go back.
    attn_implementation = "sdpa",
    # token = "YOUR_HF_TOKEN", # HF Token for gated models
)


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.8.19: Fast Gemma3 patching. Transformers: 5.5.0.
   \\   /|    NVIDIA RTX PRO 6000 Blackwell Server Edition. Num GPUs = 1. Max memory: 94.971 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 12.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: Gemma3_Text does not support SDPA - switching to fast eager.
Unsloth: Using bfloat16 full finetuning which cuts memory usage by 50%.
To enable float32 training, use `float32_mixed_precision = True` during FastLanguageModel.from_pretrained


Loading weights:   0%|          | 0/340 [00:00<?, ?it/s]

The tokenizer you are loading from '/content/drive/MyDrive/mufasa/checkpoints/gemma3-1b-cpt-v4/merged_16bit' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.


In [ ]:
from unsloth.chat_templates import get_chat_template
tokenizer = get_chat_template(
    tokenizer,
    chat_template = "gemma3",
)


In [ ]:
# ---- the MUFASA training set -------------------------------------------
# sft_mixed is the builder's superset: every strict row plus lower-tier rows,
# each carrying a verification_tier. Keep TIERS = None to train on all of it,
# or name the tiers you want.
import json

import pandas as pd
from datasets import Dataset

TIERS = None          # e.g. {"VERIFIED"} to train on the strict subset only

frame = pd.read_parquet(
    SFT_PARQUET, columns=["split", "verification_tier", "messages_json"],
)
frame = frame[frame["split"] == "train"]
if TIERS:
    frame = frame[frame["verification_tier"].isin(TIERS)]

dataset = Dataset.from_dict(
    {"conversations": [json.loads(row) for row in frame["messages_json"]]}
)
print(f"{len(dataset):,} training conversations")
print(frame["verification_tier"].value_counts().to_dict())

# ---- held-out set, so overfitting shows up DURING the run ----------------
holdout = pd.read_parquet(
    SFT_PARQUET, columns=["split", "verification_tier", "messages_json"],
)
holdout = holdout[holdout["split"] == "evaluate"]
if TIERS:
    holdout = holdout[holdout["verification_tier"].isin(TIERS)]
holdout = holdout.head(400)      # enough for a stable curve, cheap to score

eval_dataset = Dataset.from_dict(
    {"conversations": [json.loads(row) for row in holdout["messages_json"]]}
)
print(f"{len(eval_dataset):,} held-out conversations")


353,697 training conversations
{'UNVERIFIED': 229583, 'VERIFIED': 124114}
400 held-out conversations


In [ ]:
dataset[0]


{'conversations': [{'content': 'You are a research assistant for African scientific literature. Answer using only the evidence provided. If the evidence does not contain the answer, say so plainly.\n\nStudy context (scope metadata; factual support still comes from Evidence):\nStudy: \'Prevalence and effect of schistosome and soil-transmitted helminth infection on labour input in rice-growing communities of Ogun State, Nigeria\'\nDiscipline: PARASITOLOGY\nContext 1: focus: Ogun State rice-growing communities; population: The sample population was restricted to consented volunteers (adults and school age children) and young children whose parents or guardians had given their consent and are resident members of the communities.; sample: 243 consented individuals; period: May 2009 to March 2010; conditions: [{"name":"EXPERIMENTAL_SETTING","value_text":"nine rice- growing rural communities in two Local Government Areas (LGA) of Ogun State, Nigeria"},{"name":"BASELINE_STATUS","value_text":"T

In [ ]:
def formatting_prompts_func(examples):
   convos = examples["conversations"]
   texts = [tokenizer.apply_chat_template(convo, tokenize = False, add_generation_prompt = False).removeprefix('<bos>') for convo in convos]
   return { "text" : texts, }

dataset = dataset.map(formatting_prompts_func, batched = True)
# The held-out set needs the same 'text' field, or TRL cannot read it.
eval_dataset = eval_dataset.map(formatting_prompts_func, batched = True)


Map:   0%|          | 0/353697 [00:00<?, ? examples/s]

Map:   0%|          | 0/400 [00:00<?, ? examples/s]

In [ ]:
dataset[0]["text"]


'<start_of_turn>user\nYou are a research assistant for African scientific literature. Answer using only the evidence provided. If the evidence does not contain the answer, say so plainly.\n\nStudy context (scope metadata; factual support still comes from Evidence):\nStudy: \'Prevalence and effect of schistosome and soil-transmitted helminth infection on labour input in rice-growing communities of Ogun State, Nigeria\'\nDiscipline: PARASITOLOGY\nContext 1: focus: Ogun State rice-growing communities; population: The sample population was restricted to consented volunteers (adults and school age children) and young children whose parents or guardians had given their consent and are resident members of the communities.; sample: 243 consented individuals; period: May 2009 to March 2010; conditions: [{"name":"EXPERIMENTAL_SETTING","value_text":"nine rice- growing rural communities in two Local Government Areas (LGA) of Ogun State, Nigeria"},{"name":"BASELINE_STATUS","value_text":"The communi

In [ ]:
from trl import SFTTrainer, SFTConfig
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    eval_dataset = eval_dataset,
    args = SFTConfig(
        dataset_text_field = "text",
        per_device_train_batch_size = 16,
        gradient_accumulation_steps = 4,   # 64 sequences per optimiser step
        # 100 steps at batch 4 was 400 of 353,697 examples - Unsloth's demo.
        # One epoch at 64 sequences/step is ~5,500 steps. Set MAX_STEPS below
        # to stop earlier; the held-out curve will tell you when it stops paying.
        num_train_epochs = 1,
        warmup_ratio = 0.03,
        learning_rate = 2e-5, # Unsloth's own advice for long runs, and full
                              # finetuning wants a lower LR than LoRA does.
        logging_steps = 10,
        eval_strategy = "steps",
        eval_steps = 250,
        per_device_eval_batch_size = 2,
        optim = "adamw_8bit",
        weight_decay = 0.001,
        lr_scheduler_type = "linear",
        seed = 3407,
        # To Drive. 'outputs' is the VM disk and dies with the session; a run
        # this long WILL be interrupted at least once.
        output_dir = str(DRIVE / "checkpoints" / RUN_LABEL / "sft_run"),
        save_steps = 250,
        save_total_limit = 2,
        report_to = "none", # Use TrackIO/WandB etc
    ),
)


warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Unsloth: Tokenizing ["text"] (num_proc=8):   0%|          | 0/353697 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=8):   0%|          | 0/400 [00:00<?, ? examples/s]

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.


In [ ]:
from unsloth.chat_templates import train_on_responses_only
trainer = train_on_responses_only(trainer)


Unsloth: Auto-detected instruction_part = '<start_of_turn>user\n' and response_part = '<start_of_turn>model\n'


Map (num_proc=8):   0%|          | 0/353697 [00:00<?, ? examples/s]

Filter (num_proc=8):   0%|          | 0/353697 [00:00<?, ? examples/s]

Unsloth: Removed 5 out of 353697 samples from train_dataset where all labels were -100 (no response marker found, usually truncation). This prevents NaN loss during training.


Map (num_proc=8):   0%|          | 0/400 [00:00<?, ? examples/s]

In [ ]:
tokenizer.decode(trainer.train_dataset[100]["input_ids"])


'<bos><start_of_turn>user\nYou are a research assistant for African scientific literature. Answer using only the evidence provided. If the evidence does not contain the answer, say so plainly.\n\nStudy context (scope metadata; factual support still comes from Evidence):\nStudy: \'Hydrochemistry of surface water and groundwater in the shale bedrock, Cross River Basin and Niger Delta Region, Nigeria\'\nDiscipline: HYDROLOGY_HYDROGEOLOGY\nLocation: southeastern Nigeria\nContext 1: focus: Cross River Basin and Niger Delta shale-water study; design: detailed geochemical study; population: surface water (rain, streams, rivers and ponds) and groundwater (shallow hand dug wells and deep boreholes) sources; sample: Fifty-two samples; period: between July and August 2009; conditions: [{"name":"EXPERIMENTAL_SETTING","value_text":"areas underlain by shale bedrock in parts of southeastern Nigeria"},{"name":"SAMPLING_SETTING","value_text":"surface water (rain, streams, rivers and ponds) and groundwa

In [ ]:
tokenizer.decode([tokenizer.pad_token_id if x == -100 else x for x in trainer.train_dataset[100]["labels"]]).replace(tokenizer.pad_token, " ")


'                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                     More than 60 % of the area of study is underlain by shale which is often problematic hydrogeologically mainly due to low permeability.\n\nAnswer: The large shale extent matters because its low permeability make

In [ ]:
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")


GPU = NVIDIA RTX PRO 6000 Blackwell Server Edition. Max memory = 94.971 GB.
1.904 GB of memory reserved.


In [ ]:
# ---- avoid the flex_attention backward kernel ----------------------------
# Gemma 3's sliding-window attention compiles to a flex_attention Triton kernel
# whose backward wants ~112 KB of shared memory PER THREAD BLOCK:
#
#   InductorError: No valid triton configs. OutOfMemoryError: out of resource:
#   triton_tem_fused_flex_attention_backward  Required: 114688  limit: 101376
#
# 101,376 bytes is the per-block ceiling on Ada AND on workstation Blackwell
# (sm_120) alike - the 228 KB figure quoted for Blackwell is per SM, which is
# not what this kernel is limited by. Only A100 (164 KB) and H100 (227 KB)
# clear it, so this is not something a different consumer card fixes.
#
# suppress_errors makes Inductor fall back to eager for kernels it cannot fit
# rather than raising. Slower on those kernels, but the run completes.
import torch._dynamo

torch._dynamo.config.suppress_errors = True
print("Inductor will fall back to eager for kernels that do not fit")


Inductor will fall back to eager for kernels that do not fit


In [15]:
# Resumes from the Drive checkpoint if an earlier session was cut short.
resume = any(Path(trainer.args.output_dir).glob("checkpoint-*"))
print("resuming" if resume else "starting fresh")
trainer_stats = trainer.train(resume_from_checkpoint=resume)


starting fresh


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 353,692 | Num Epochs = 1 | Total steps = 5,527
O^O/ \_/ \    Batch size per device = 16 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (16 x 4 x 1) = 64
 "-____-"     Trainable parameters = 999,885,952 of 999,885,952 (100.00% trained)


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss,Validation Loss
250,0.746675,0.603733
500,0.727917,0.595138
750,0.717932,0.595055


Filter:   0%|          | 0/400 [00:00<?, ? examples/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/mufasa/checkpoints/gemma3-1b-cpt-v4/sft_run/checkpoint-1250/tokenizer_config.json.
Unsloth: Preserved sentencepiece asset `tokenizer.model` in /content/drive/MyDrive/mufasa/checkpoints/gemma3-1b-cpt-v4/sft_run/checkpoint-1250.


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/mufasa/checkpoints/gemma3-1b-cpt-v4/sft_run/checkpoint-1500/tokenizer_config.json.
Unsloth: Preserved sentencepiece asset `tokenizer.model` in /content/drive/MyDrive/mufasa/checkpoints/gemma3-1b-cpt-v4/sft_run/checkpoint-1500.


KeyboardInterrupt: 

In [ ]:
used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
used_memory_for_training = round(used_memory - start_gpu_memory, 3)
used_percentage = round(used_memory / max_memory * 100, 3)
training_percentage = round(used_memory_for_training / max_memory * 100, 3)
print(f"{trainer_stats.metrics['train_runtime']} seconds used for training.")
print(
    f"{round(trainer_stats.metrics['train_runtime']/60, 2)} minutes used for training."
)
print(f"Peak reserved memory = {used_memory} GB.")
print(f"Peak reserved memory for training = {used_memory_for_training} GB.")
print(f"Peak reserved memory % of max memory = {used_percentage} %.")
print(f"Peak reserved memory for training % of max memory = {training_percentage} %.")


In [ ]:
messages = [
    {"role" : "user", "content" : dataset["conversations"][0][0]["content"]}
]
text = tokenizer.apply_chat_template(
    messages,
    tokenize = False,
    add_generation_prompt = True, # Must add for generation
).removeprefix('<bos>')

from transformers import TextStreamer
_ = model.generate(
    **tokenizer(text, return_tensors = "pt").to("cuda"),
    max_new_tokens = 256,
    temperature = 1, top_p = 0.95, top_k = 64,
    streamer = TextStreamer(tokenizer, skip_prompt = True),
)


In [ ]:
# ---- save the finetuned model -------------------------------------------
# Full finetuning has no adapter: `model` already holds the complete weights,
# so save_pretrained writes something directly loadable. The peft_config branch
# only matters if full_finetuning is ever set back to False - then the adapter
# has to be merged into the base before anything else can use it.
SAVE_DIR = DRIVE / "checkpoints" / "gemma3-1b-cpt-v4" / "sft"
SAVE_DIR.parent.mkdir(parents=True, exist_ok=True)

if getattr(model, "peft_config", None):
    print("LoRA adapter detected - merging to 16-bit")
    model.save_pretrained_merged(str(SAVE_DIR), tokenizer, save_method="merged_16bit")
else:
    print("full finetune - saving complete weights")
    model.save_pretrained(str(SAVE_DIR))
    tokenizer.save_pretrained(str(SAVE_DIR))

written = sorted(p.name for p in SAVE_DIR.iterdir()) if SAVE_DIR.is_dir() else []
size = sum(p.stat().st_size for p in SAVE_DIR.iterdir() if p.is_file()) / 1e9
print(f"saved to {SAVE_DIR}  ({size:,.2f} GB)")
print("  " + ", ".join(written[:8]))


In [ ]:
# ---- GGUF: the file the judges actually run ------------------------------
# Written from SAVE_DIR rather than from memory, so this cell can be re-run in
# a fresh session without repeating the training.
EXPORT_GGUF = True
GGUF_METHOD = "Q8_0"      # Q8_0, BF16 and F16 are supported

if EXPORT_GGUF:
    GGUF_DIR = DRIVE / "checkpoints" / "gemma3-1b-cpt-v4" / "gguf"
    GGUF_DIR.parent.mkdir(parents=True, exist_ok=True)
    model.save_pretrained_gguf(
        str(GGUF_DIR), tokenizer, quantization_method=GGUF_METHOD,
    )
    files = sorted(GGUF_DIR.rglob("*.gguf")) if GGUF_DIR.is_dir() else []
    for path in files:
        print(f"{path.stat().st_size/1e9:,.2f} GB  {path}")
    if not files:
        print("no .gguf produced - check the output above for the conversion log")


## Evaluating the SFT model

Five measurements, against the CPT checkpoint this run started from.

| measurement | direction | what it tells you |
|---|---|---|
| **held-out loss** | lower | the only overfitting signal; produced during training |
| **exact match / token F1** | higher | short-answer correctness, SQuAD convention, on the answer span alone |
| **numeric / entity accuracy** | higher | are the *figures and places* right - what similarity metrics cannot see |
| **citation accuracy** | higher | the fabrication rate, measured against the builder's true author-year |
| **format compliance** | higher | does it emit the Provenance / Citation / Study-basis contract at all |
| **judge** (optional) | higher | correctness, grounding, usefulness, 1-5, via OpenRouter |

The combination that matters most: **high token F1 with low numeric accuracy**
means the answers read well and the numbers are wrong. That is the failure this
harness exists to catch, and the one a BLEU or BERTScore run would hide.

The CPT base has never seen the output format, so it will score near zero on
citation and format. That is a real result, but report it as *the SFT taught the
contract*, not as a like-for-like quality gap.


In [ ]:
# ========================= evaluation toolkit ================================
# Self-contained: no external module. Five measurements against the CPT
# checkpoint this run started from.
import json
import re
import string
import textwrap
import threading
from collections import Counter
from concurrent.futures import ThreadPoolExecutor, as_completed

import pandas as pd

PROVENANCE = re.compile(r"\n\s*Provenance\s*:", re.I)
CITATION = re.compile(r"^\s*Citation\s*:\s*\(?([^)\n]+?)\)?\s*(?:\[unverified\])?\s*$",
                      re.I | re.M)
BASIS = re.compile(r"^\s*Study basis\s*:", re.I | re.M)
LEAD_ANSWER = re.compile(r".*\bAnswer\s*:\s*", re.S)
NUMBER = re.compile(r"\d+(?:\.\d+)?")
# Multi-word proper nouns only. Single capitalised words are ignored: any word
# can start a sentence, so that test is not robust.
ENTITY = re.compile(r"\b[A-Z][a-z]{2,}(?:\s+[A-Z][a-z]{2,})+\b")
ARTICLES = re.compile(r"\b(a|an|the)\b")
PUNCT = str.maketrans("", "", string.punctuation)


def answer_span(text):
    """The answer alone: no provenance block, no reasoning preamble."""
    head = PROVENANCE.split(str(text or ""), maxsplit=1)[0]
    if re.search(r"\bAnswer\s*:", head):
        head = LEAD_ANSWER.sub("", head, count=1)
    return head.strip()


def citation_of(text):
    found = CITATION.search(str(text or ""))
    return found.group(1).strip() if found else ""


def has_format(text):
    text = str(text or "")
    return bool(PROVENANCE.search(text)) and bool(CITATION.search(text)) \
        and bool(BASIS.search(text))


def normalise(text):
    text = str(text or "").lower().translate(PUNCT)
    return " ".join(ARTICLES.sub(" ", text).split())


def exact_match(reference, prediction):
    return float(normalise(reference) == normalise(prediction))


def token_f1(reference, prediction):
    gold, pred = normalise(reference).split(), normalise(prediction).split()
    if not gold or not pred:
        return float(gold == pred)
    overlap = sum((Counter(gold) & Counter(pred)).values())
    if not overlap:
        return 0.0
    precision, recall = overlap / len(pred), overlap / len(gold)
    return 2 * precision * recall / (precision + recall)


def numeric_accuracy(reference, prediction, tolerance=0.01):
    """Share of the reference's figures reproduced.

    Integers must match exactly. A relative tolerance on a year is nonsense -
    1% of 2009 is twenty years - so the tolerance applies only to decimals,
    where it absorbs rounding in a measured quantity.
    """
    want = NUMBER.findall(str(reference or ""))
    if not want:
        return None
    got = [float(x) for x in NUMBER.findall(str(prediction or ""))]
    hits = 0
    for raw in want:
        target = float(raw)
        margin = abs(target) * tolerance if "." in raw else 0.0
        if any(abs(value - target) <= margin for value in got):
            hits += 1
    return hits / len(want)


def entity_accuracy(reference, prediction):
    want = set(ENTITY.findall(str(reference or "")))
    if not want:
        return None
    body = str(prediction or "")
    return sum(1 for item in want if item in body) / len(want)


def citation_accuracy(reference, prediction):
    truth = normalise(citation_of(reference))
    if not truth:
        return None
    return float(normalise(citation_of(prediction)) == truth)


def generate_answers(a_model, a_tokenizer, prompts, max_new_tokens=320,
                     batch_size=8):
    """Greedy decoding, batched, so the score is reproducible."""
    import torch
    from tqdm.auto import tqdm

    a_model.eval()
    if a_tokenizer.pad_token_id is None:
        a_tokenizer.pad_token = a_tokenizer.eos_token
    side = a_tokenizer.padding_side
    a_tokenizer.padding_side = "left"
    out = []
    try:
        for start in tqdm(range(0, len(prompts), batch_size),
                          desc="generating", unit="batch"):
            chunk = prompts[start:start + batch_size]
            batch = a_tokenizer(chunk, return_tensors="pt", padding=True,
                                truncation=True, max_length=4096).to(a_model.device)
            with torch.no_grad():
                produced = a_model.generate(
                    **batch, max_new_tokens=max_new_tokens, do_sample=False,
                    pad_token_id=a_tokenizer.pad_token_id,
                )
            for row, source in zip(produced, batch["input_ids"]):
                out.append(a_tokenizer.decode(row[len(source):],
                                              skip_special_tokens=True).strip())
    finally:
        a_tokenizer.padding_side = side
    return out


JUDGE_PROMPT = """You are grading a scientific assistant's answer against the reference.

QUESTION:
{question}

REFERENCE ANSWER:
{reference}

MODEL ANSWER:
{prediction}

Score the MODEL ANSWER on three axes, 1 to 5.
  correctness  - does it state the same facts, figures and places as the reference
  grounding    - are its claims supported, with nothing invented
  usefulness   - would a researcher be served by this answer

Reply with JSON only: {{"correctness": n, "grounding": n, "usefulness": n}}"""


class KeyPool:
    """Round-robin over the pasted keys, thread-safe."""

    def __init__(self, keys):
        self.keys = [k.strip() for k in (keys or []) if k and k.strip()]
        self._at = 0
        self._lock = threading.Lock()

    def __bool__(self):
        return bool(self.keys)

    def next(self):
        with self._lock:
            key = self.keys[self._at % len(self.keys)]
            self._at += 1
            return key


def _judge_one(pool, model_id, question, reference, prediction, timeout):
    import urllib.request

    body = json.dumps({
        "model": model_id,
        "messages": [{"role": "user", "content": JUDGE_PROMPT.format(
            question=str(question)[:2000], reference=str(reference)[:2000],
            prediction=str(prediction)[:2000])}],
        "temperature": 0,
        "max_tokens": 120,
    }).encode()
    request = urllib.request.Request(
        "https://openrouter.ai/api/v1/chat/completions", data=body,
        headers={"Authorization": f"Bearer {pool.next()}",
                 "Content-Type": "application/json"},
    )
    try:
        with urllib.request.urlopen(request, timeout=timeout) as response:
            payload = json.loads(response.read())
        text = payload["choices"][0]["message"]["content"]
        found = re.search(r"\{.*\}", text, re.S)
        scores = json.loads(found.group(0)) if found else {}
        return {axis: float(scores[axis]) for axis in
                ("correctness", "grounding", "usefulness") if axis in scores}
    except Exception:
        return {}


def judge_answers(frame, keys, model_id="openai/gpt-oss-120b", workers=16,
                  timeout=90):
    """Grade every row, or return the frame untouched when judging is off.

    An empty key list is a supported configuration, not an error: the tables
    below simply have no judge columns.
    """
    from tqdm.auto import tqdm

    pool = KeyPool(keys)
    if not pool:
        print("judge off - no API keys supplied")
        return frame
    frame = frame.copy()
    results = {}
    workers = max(1, min(workers, len(pool.keys) * 8))
    with ThreadPoolExecutor(max_workers=workers) as runner:
        futures = {
            runner.submit(_judge_one, pool, model_id, row.question,
                          row.reference, row.prediction, timeout): row.Index
            for row in frame.itertuples(index=True)
        }
        for future in tqdm(as_completed(futures), total=len(futures),
                           desc="judging", unit="answer"):
            results[futures[future]] = future.result()
    for axis in ("correctness", "grounding", "usefulness"):
        frame[f"judge_{axis}"] = [results.get(i, {}).get(axis) for i in frame.index]
    print(f"judged {frame['judge_correctness'].notna().sum():,}/{len(frame):,} "
          f"answers with {len(pool.keys)} key(s)")
    return frame


METRICS = ["exact_match", "token_f1", "numeric_accuracy", "entity_accuracy",
           "citation_accuracy", "format_compliance",
           "judge_correctness", "judge_grounding", "judge_usefulness"]


def score_predictions(frame):
    """Per-row metrics. Needs question, reference, prediction."""
    frame = frame.copy()
    gold = frame["reference"].map(answer_span)
    pred = frame["prediction"].map(answer_span)
    frame["exact_match"] = [exact_match(r, p) for r, p in zip(gold, pred)]
    frame["token_f1"] = [token_f1(r, p) for r, p in zip(gold, pred)]
    frame["numeric_accuracy"] = [numeric_accuracy(r, p) for r, p in zip(gold, pred)]
    frame["entity_accuracy"] = [entity_accuracy(r, p) for r, p in zip(gold, pred)]
    frame["citation_accuracy"] = [
        citation_accuracy(r, p)
        for r, p in zip(frame["reference"], frame["prediction"])
    ]
    frame["format_compliance"] = frame["prediction"].map(has_format).astype(float)
    return frame


def summarise_scores(scored, name="SFT"):
    out = {}
    for metric in METRICS:
        if metric in scored.columns:
            values = pd.to_numeric(scored[metric], errors="coerce").dropna()
            if len(values):
                out[metric] = float(values.mean())
    return pd.Series(out, name=name)


def compare_scores(scores_by_name):
    """One table, one column per model. Pass an ordered dict base -> CPT -> SFT.

    The change column is last minus first, so it reads as what the whole
    pipeline bought over the untouched base.
    """
    table = pd.DataFrame(dict(scores_by_name))
    names = list(scores_by_name)
    table["change"] = table[names[-1]] - table[names[0]]
    table["better?"] = ["yes" if v > 0 else "no" for v in table["change"]]
    return table.reset_index().rename(columns={"index": "metric"})


def read_out(table, tuned_name="SFT"):
    """What the numbers mean, including the combination that misleads."""
    lookup = table.set_index("metric")

    def get(metric):
        return float(lookup.loc[metric, tuned_name]) if metric in lookup.index else float("nan")

    lines = []
    fmt, cite = get("format_compliance"), get("citation_accuracy")
    num, ent, f1 = get("numeric_accuracy"), get("entity_accuracy"), get("token_f1")

    if fmt == fmt:
        if fmt > 0.95:
            lines.append(f"Format compliance {fmt:.0%} - the model reliably emits the "
                         "provenance contract.")
        else:
            lines.append(f"Format compliance only {fmt:.0%}. {1-fmt:.0%} of answers omit "
                         "part of the Provenance/Citation/Study-basis block; train "
                         "longer, or check train_on_responses_only masked correctly.")
    if cite == cite:
        if cite >= 0.999:
            lines.append("Citation accuracy 100% - no fabricated author-year in this "
                         "sample. Check the sample size before quoting that.")
        else:
            lines.append(f"Citation accuracy {cite:.0%}. The other {1-cite:.0%} are "
                         "fabrications - a plausible author-year on the wrong study. "
                         "This is the number to quote when asked about hallucination.")
    if num == num and f1 == f1 and f1 - num > 0.15:
        lines.append(f"Token F1 ({f1:.2f}) sits well above numeric accuracy ({num:.2f}). "
                     "The answers read correctly and the figures are wrong - exactly the "
                     "failure a similarity metric would have hidden.")
    elif num == num:
        tail = f", entity accuracy {ent:.0%}" if ent == ent else ""
        lines.append(f"Numeric accuracy {num:.0%}{tail} - figures and places are being "
                     "reproduced, not just the prose.")
    if "judge_correctness" in lookup.index:
        judged = get("judge_correctness")
        if judged == judged:
            lines.append(f"Judge correctness {judged:.2f}/5. Read it beside citation "
                         "accuracy: a high judge score with poor citations means fluent "
                         "answers attributed to the wrong papers.")
    return lines


SHADES = ["#c3c9d4", "#7d93ad", "#2f5d8c", "#1d3a57"]


def plot_scores(table, names, title="base vs CPT vs SFT"):
    """One bar per model per metric, plus the end-to-end change beside them.

    Judge axes are 1-5 while everything else is 0-1, so they are rescaled for
    the chart only - the table keeps the raw values.
    """
    import matplotlib.pyplot as plt
    import numpy as np

    names = list(names)
    shown = table.copy()
    judged = shown["metric"].str.startswith("judge_")
    for column in [*names, "change"]:
        shown.loc[judged, column] = shown.loc[judged, column] / 5.0

    metrics = list(shown["metric"])
    figure, axes = plt.subplots(1, 2, figsize=(14.5, 4.6),
                                gridspec_kw={"width_ratios": [3.2, 2]})
    x = np.arange(len(metrics))
    width = 0.8 / max(len(names), 1)
    for position, name in enumerate(names):
        offset = (position - (len(names) - 1) / 2) * width
        axes[0].bar(x + offset, shown[name], width, label=name,
                    color=SHADES[position % len(SHADES)])
    axes[0].set_xticks(x)
    axes[0].set_xticklabels([m.replace("_", "\n") for m in metrics], fontsize=8)
    axes[0].set_ylim(0, 1.05)
    axes[0].set_title(f"{title} - higher is better (judge axes /5)", fontsize=10)
    axes[0].legend(fontsize=8)
    axes[0].grid(axis="y", alpha=0.25)

    axes[1].barh(metrics, shown["change"],
                 color=["#3e7a5e" if v > 0 else "#a34434" for v in shown["change"]])
    axes[1].axvline(0, color="#333", linewidth=0.8)
    axes[1].set_title(f"{names[-1]} minus {names[0]} (green = improved)", fontsize=10)
    axes[1].tick_params(labelsize=8)
    axes[1].grid(axis="x", alpha=0.25)
    plt.tight_layout()
    return figure


print("evaluation toolkit ready")


In [ ]:
# ===================== generate answers with the SFT model ===================
N_EVAL = 300          # answers scored; 300 is enough for the rates to settle
GEN_BATCH = 8         # lower if generation OOMs

sample = pd.read_parquet(SFT_PARQUET, columns=["split", "prompt", "response"])
sample = sample[sample["split"] == "evaluate"]
sample = sample.sample(n=min(N_EVAL, len(sample)), random_state=7)

def as_chat(question, a_tokenizer):
    text = a_tokenizer.apply_chat_template(
        [{"role": "user", "content": question}],
        tokenize=False, add_generation_prompt=True,
    )
    return text.removeprefix(a_tokenizer.bos_token or "")

prompts = [as_chat(q, tokenizer) for q in sample["prompt"]]
sft_frame = pd.DataFrame({
    "question": sample["prompt"].tolist(),
    "reference": sample["response"].tolist(),
    "prediction": generate_answers(model, tokenizer, prompts, batch_size=GEN_BATCH),
})
print(sft_frame["prediction"].iloc[0][:400])


In [ ]:
# ============== the same questions, on the two earlier models ================
# Three models are compared: the untouched base, the CPT checkpoint, and the
# SFT model just trained. They are run one at a time and freed in between -
# three models will not sit on one GPU together. The SFT weights are already
# saved, so freeing it costs nothing.
import gc

import torch

# The stock checkpoint each CPT run started from.
ORIGINAL_BASE = "unsloth/gemma-3-1b-pt"

del model, trainer
gc.collect()
torch.cuda.empty_cache()

def answers_from(model_id, label):
    global loaded, loaded_tokenizer
    print(f"\n=== {label}: {model_id}")
    loaded, loaded_tokenizer = MODEL_API.from_pretrained(
        model_name = str(model_id),
        max_seq_length = max_seq_length,
        load_in_4bit = False,
        full_finetuning = False,
    )
    prompts_here = [as_chat(q, loaded_tokenizer) for q in sft_frame["question"]]
    produced = generate_answers(loaded, loaded_tokenizer, prompts_here,
                                batch_size=GEN_BATCH)
    frame = pd.DataFrame({
        "question": sft_frame["question"],
        "reference": sft_frame["reference"],
        "prediction": produced,
    })
    del loaded, loaded_tokenizer
    gc.collect()
    torch.cuda.empty_cache()
    return frame

cpt_frame = answers_from(CPT_CHECKPOINT, "CPT")
base_frame = answers_from(ORIGINAL_BASE, "base")


In [ ]:
# ============================ LLM judge (optional) ===========================
# Paste one key per line. Leave the list EMPTY to skip judging entirely - the
# tables below simply lose their judge columns; nothing else changes.
#
# Keys are used round-robin across threads, so more keys means more throughput.
OPENROUTER_KEYS = [
    # "sk-or-v1-...",
    # "sk-or-v1-...",
]
JUDGE_MODEL = "openai/gpt-oss-120b"
JUDGE_WORKERS = 16      # capped internally at 8 x number of keys

sft_frame = judge_answers(sft_frame, OPENROUTER_KEYS, model_id=JUDGE_MODEL,
                          workers=JUDGE_WORKERS)
cpt_frame = judge_answers(cpt_frame, OPENROUTER_KEYS, model_id=JUDGE_MODEL,
                          workers=JUDGE_WORKERS)
base_frame = judge_answers(base_frame, OPENROUTER_KEYS, model_id=JUDGE_MODEL,
                           workers=JUDGE_WORKERS)


In [ ]:
# ========================== score, tabulate, chart ===========================
base_scored = score_predictions(base_frame)
cpt_scored = score_predictions(cpt_frame)
sft_scored = score_predictions(sft_frame)

NAMES = ["base", "CPT", "SFT"]
table = compare_scores({
    "base": summarise_scores(base_scored, "base"),
    "CPT":  summarise_scores(cpt_scored, "CPT"),
    "SFT":  summarise_scores(sft_scored, "SFT"),
})
display(table.round(4))
display(plot_scores(table, NAMES, title=f"{RUN_LABEL} - base vs CPT vs SFT"))

print()
print("What this says")
print("=" * 74)
for line in read_out(table):
    print(textwrap.fill(line, 88, initial_indent="  - ", subsequent_indent="    "))

out = DRIVE / "checkpoints" / RUN_LABEL / "sft_evaluation.csv"
table.to_csv(out, index=False)
sft_scored.to_csv(out.with_name("sft_predictions.csv"), index=False)
print()
print("saved", out)


In [ ]:
# ================ capability regression, lm-evaluation-harness ===============
# The standard public benchmarks, run identically on all three models. This is
# the "did any stage damage the model" check, and the one judges recognise.
# Roughly 20-40 minutes per model.
RUN_HARNESS = False

TARGETS = {
    "base": ORIGINAL_BASE,
    "cpt":  str(CPT_CHECKPOINT),
    "sft":  str(DRIVE / "checkpoints" / RUN_LABEL / "sft"),
}

if RUN_HARNESS:
    !pip install -q lm-eval
    for label, target in TARGETS.items():
        out_path = str(DRIVE / "checkpoints" / RUN_LABEL / ("lmeval_" + label))
        !lm_eval --model hf --model_args pretrained={target} \
            --tasks arc_easy,arc_challenge,truthfulqa_mc2,mmlu_clinical_knowledge,mmlu_college_biology \
            --device cuda:0 --batch_size 8 --limit 500 --output_path {out_path}


In [ ]:
# ============ evaluate the BEST checkpoint from an interrupted run ===========
# The SFT run was stopped once held-out loss turned upward. This picks the
# checkpoint with the lowest eval_loss that still exists on disk, loads it, and
# runs the same comparison as the cells above.
#
# save_total_limit keeps only the most recent checkpoints, so the true optimum
# may already have been deleted; this reports what it actually found.
import gc
import json
from pathlib import Path

import torch

SFT_RUN = DRIVE / "checkpoints" / RUN_LABEL / "sft_run"
found = sorted(SFT_RUN.glob("checkpoint-*"), key=lambda p: int(p.name.split("-")[1]))
print("checkpoints on disk:", [p.name for p in found] or "NONE")

state = json.loads((found[-1] / "trainer_state.json").read_text(encoding="utf-8"))
history = [(e["step"], e["eval_loss"]) for e in state["log_history"] if "eval_loss" in e]
print()
print("step    eval_loss")
for step, loss in history:
    mark = ""
    print(f"{step:>5}   {loss:.6f}{mark}")

alive = {int(p.name.split("-")[1]): p for p in found}
usable = [(loss, step) for step, loss in history if step in alive]
if usable:
    best_loss, best_step = min(usable)
    BEST = alive[best_step]
    optimum = min(loss for _, loss in history)
    print(f"\nbest surviving : {BEST.name}  (eval_loss {best_loss:.6f})")
    if best_loss > optimum + 1e-9:
        print(f"note           : the run's true optimum was {optimum:.6f}, "
              "on a checkpoint save_total_limit has already deleted")
else:
    BEST = found[-1]
    print(f"\nno eval matched a surviving checkpoint - using {BEST.name}")

# a Trainer checkpoint normally carries the tokenizer; top it up if not
if not any(p.name.startswith("tokenizer") for p in BEST.iterdir()):
    import shutil
    for item in (CPT_CHECKPOINT).iterdir():
        if item.name.startswith(("tokenizer", "special_tokens")):
            shutil.copy2(item, BEST / item.name)
    print("tokenizer copied from the CPT checkpoint")

# free whatever is loaded, then score the best checkpoint
for name in ("model", "trainer", "base_model"):
    if name in dir():
        exec(f"del {name}")
gc.collect()
torch.cuda.empty_cache()

sft_model, sft_tokenizer = MODEL_API.from_pretrained(
    model_name = str(BEST),
    max_seq_length = max_seq_length,
    load_in_4bit = False,
    full_finetuning = False,
)
best_prompts = [as_chat(q, sft_tokenizer) for q in sft_frame["question"]]
best_frame = pd.DataFrame({
    "question": sft_frame["question"],
    "reference": sft_frame["reference"],
    "prediction": generate_answers(sft_model, sft_tokenizer, best_prompts,
                                   batch_size=GEN_BATCH),
})
best_frame = judge_answers(best_frame, OPENROUTER_KEYS, model_id=JUDGE_MODEL,
                           workers=JUDGE_WORKERS)

table = compare_scores({
    "base": summarise_scores(score_predictions(base_frame), "base"),
    "CPT":  summarise_scores(score_predictions(cpt_frame), "CPT"),
    "SFT":  summarise_scores(score_predictions(best_frame), "SFT"),
})
display(table.round(4))
display(plot_scores(table, ["base", "CPT", "SFT"],
                    title=f"{RUN_LABEL} - {BEST.name}"))
for line in read_out(table):
    print(textwrap.fill(line, 88, initial_indent="  - ", subsequent_indent="    "))

table.to_csv(DRIVE / "checkpoints" / RUN_LABEL / "sft_evaluation.csv", index=False)
print("\nsaved sft_evaluation.csv")
